In [1]:
%load_ext dotenv
%dotenv

In [3]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # force synchronous debugging

from datetime import datetime
from pathlib import Path
import logging
from huggingface_hub import login, whoami
from transformers.utils.logging import disable_progress_bar

from gsm_benchmarker.dataset_wrapper import GSMSymbolicDataset
from gsm_benchmarker.benchmark_config import BenchmarkConfig
from gsm_benchmarker.benchmark import BenchmarkRunner
from gsm_benchmarker.models_config_parser import ModelsConfig
from gsm_benchmarker.utils.logging_setup import install_colored_logger, setup_log_file_handler
from gsm_benchmarker.utils.seeds import set_seed


disable_progress_bar()
set_seed(42)


hf_api_token = os.environ['HUGGINGFACEHUB_API_TOKEN']
login(hf_api_token)

whoami()['name']

openai package not installed; OpenAI models will not be available
anthropic package not installed; Anthropic models will not be available
google-genai package not installed; Gemini family models will not be available


'the-mysh'

In [4]:
OUTPUT_ROOT_PATH = Path(f"../../../data/gsm-symbolic").resolve()
RESULTS_ROOT_PATH = OUTPUT_ROOT_PATH / f"outputs/{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(OUTPUT_ROOT_PATH)
print(RESULTS_ROOT_PATH)
print(os.environ['HF_HOME'])

/home/guests2/dkd/data/gsm-symbolic
/home/guests2/dkd/data/gsm-symbolic/outputs/20251105_114302
/media/generalstorage4/dkdstorage


In [ ]:
for log_name in ('urllib3', 'fsspec', 'filelock', 'h5py', 'httpcore', 'httpx', 'google_genai', 'jax', 'root', 'bitsandbytes', 'transformers_modules'):
    logging.getLogger(log_name).setLevel(logging.WARNING)

install_colored_logger(level=logging.DEBUG)
setup_log_file_handler(OUTPUT_ROOT_PATH / "logs")

In [ ]:
models_config = ModelsConfig()
all_models = models_config.all_models
for m in all_models:
    print(m.name, m.api_type, m.extra_kwargs)  # note: openai - insufficient quota for benchmarking

In [ ]:
open_models = models_config.open_models
print([m.name for m in open_models])

In [ ]:
variants = GSMSymbolicDataset.Variant

In [ ]:

br = BenchmarkRunner(
    models=open_models,
    dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
    storage_path=RESULTS_ROOT_PATH,
    config=BenchmarkConfig(trust_remote_code_global=True, gpu0_max_memory="46GiB", cpu_max_memory="100GiB")
)

In [ ]:
br.run()

In [ ]:
# failed_models = [models_config['microsoft/Phi-3-small-128k-instruct']]  #, 'mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3']
# br2 = BenchmarkRunner(
#     models=failed_models,
#     dset_variants=[variants.GSM8K, variants.main, variants.p1, variants.p2],
#     storage_path=RESULTS_ROOT_PATH,
#     config=BenchmarkConfig(trust_remote_code_global=True, gpu0_max_memory="46GiB", cpu_max_memory="100GiB")
# )
# br2.run(n_sets=2, n_per_set=2)

In [ ]:
br.summarise_results()

In [ ]:
el = br.results[GSMSymbolicDataset.Variant.main][open_models[-1].name].loc[0, 0]
el

In [ ]:
print(el.question)
print()
print(el.response)